In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix,classification_report)
from sklearn.ensemble import ( RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

In [6]:
df = pd.read_csv(r'D:\Data_D\TMU\MRP\1.0.Dataset\Crime_Socio_Economic_Weather_Mobility.csv')

In [ ]:
print(df.shape)
df.head()

(7970658, 36)


,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,community_area,FBI Code,year,...,temp_avg,is_rain,is_snow,temp_category,high_wind,day_type,bus,rail_boardings,total_rides,day
0,29,92,161,0,0,511,5.0,49,3,2008,...,62.5,1,0,1,0,0,749048,350884,1099932,17
1,29,470,190,0,1,512,5.0,49,3,2008,...,46.0,0,0,2,1,2,1026141,626634,1652775,27
2,29,92,178,0,0,523,5.0,53,3,2008,...,78.0,0,0,1,0,2,997379,673384,1670763,5
3,3,244,162,0,0,1622,16.0,11,6,2008,...,46.5,1,0,2,1,0,510369,254291,764660,27
4,22,37,161,0,1,935,9.0,61,19,2004,...,61.5,0,0,1,0,2,1047045,625246,1672291,17


In [ ]:
X = df.drop(columns=["Arrest"])
y = df["Arrest"]

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns
print("Categorical Columns:")
print(categorical_cols)

print("Total:", len(categorical_cols))

Categorical Columns:
Index(['community_name'], dtype='object')

Total: 1


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training")

print(y_train.value_counts(normalize=True)*100)

print("Testing")

print(y_test.value_counts(normalize=True)*100)

Training
Arrest
0    74.195291
1    25.804709
Name: proportion, dtype: float64

Testing
Arrest
0    74.195299
1    25.804701
Name: proportion, dtype: float64


In [22]:
models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Decision Tree":
        DecisionTreeClassifier(random_state=42)

}

In [ ]:
results = []

for name, model in models.items():
    print(name)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]
    accuracy = accuracy_score(y_test,pred)
    precision = precision_score(y_test,pred)
    recall = recall_score(y_test,pred)
    f1 = f1_score(y_test,pred)
    roc = roc_auc_score(y_test,prob)
    pr = average_precision_score(y_test,prob)
    print(classification_report(y_test,pred))
    print(confusion_matrix(y_test,pred))

    results.append({
        "Model":name,
        "Accuracy":accuracy,
        "Precision":precision,
        "Recall":recall,
        "F1 Score":f1,
        "ROC AUC":roc,
        "PR AUC":pr
    })

Logistic Regression


C:\Users\devan\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           0       0.80      0.96      0.87   1182771
           1       0.71      0.32      0.44    411361

    accuracy                           0.79   1594132
   macro avg       0.76      0.64      0.65   1594132
weighted avg       0.78      0.79      0.76   1594132

[[1130038   52733]
 [ 281213  130148]]
Decision Tree
              precision    recall  f1-score   support

           0       0.89      0.88      0.89   1182771
           1       0.67      0.69      0.68    411361

    accuracy                           0.83   1594132
   macro avg       0.78      0.79      0.78   1594132
weighted avg       0.83      0.83      0.83   1594132

[[1039541  143230]
 [ 125914  285447]]


In [ ]:
results = pd.DataFrame(results)
results = results.sort_values(
    by="F1 Score",
    ascending=False
)

results

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,PR AUC
1,Decision Tree,0.831166,0.665879,0.693909,0.679605,0.786563,0.541217
0,Logistic Regression,0.790515,0.711654,0.316384,0.438030,0.731849,0.543362
